# Sesión 18 — Autoencoders Variacionales y GAN
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo V · Arquitecturas Avanzadas y AI Generativa**

## Objetivos de aprendizaje

1. Derivar el objetivo del VAE (ELBO) a partir de la cota inferior variacional sobre la log-verosimilitud.
2. Implementar el **truco de reparametrización** y comprender por qué permite la retropropagación.
3. Entrenar un VAE en señales fisiológicas y explorar el **espacio latente**.
4. Derivar el objetivo minimax de la GAN y comprender el colapso de modo.
5. Aplicar una GAN condicional (cGAN) a la **aumentación de datos de EEG** para BCI.

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Kingma, D.P. & Welling, M. (2014). Auto-encoding variational Bayes. *ICLR*. |
| ★★★ | Goodfellow, I. et al. (2014). Generative adversarial nets. *NeurIPS*. |
| ★★☆ | Higgins, I. et al. (2017). β-VAE: learning basic visual concepts with a constrained variational framework. *ICLR*. |
| ★★☆ | Hartmann, K.G. et al. (2018). EEG-GAN: generative adversarial networks for electroencephalographic data augmentation. *arXiv*. |
| ★★☆ | Luo, Y. & Durrani, T.S. (2020). EEG signal augmentation using conditional GANs for brain-computer interfaces. *EMBC*. |
| ★☆☆ | Blog de VAE de Lilian Weng: https://lilianweng.github.io/posts/2018-08-12-vae/ |

## Parte 0 — Configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA

rng    = np.random.default_rng(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True,
                     'grid.alpha': 0.3, 'font.size': 11})
print(f'Dispositivo: {device}')

## Parte 1 — VAE: teoría y derivación

Queremos modelar $p_\theta(\mathbf{x})$ con una variable latente $\mathbf{z}$:
$$p_\theta(\mathbf{x}) = \int p_\theta(\mathbf{x}|\mathbf{z})\,p(\mathbf{z})\,d\mathbf{z}$$

Esto es intratable directamente, así que maximizamos la **Cota Inferior de la Evidencia (ELBO)**:
$$\log p_\theta(\mathbf{x}) \ge \underbrace{\mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x})}[\log p_\theta(\mathbf{x}|\mathbf{z})]}_{\text{reconstrucción}} - \underbrace{D_{\text{KL}}(q_\phi(\mathbf{z}|\mathbf{x})\|p(\mathbf{z}))}_{\text{regularización}}$$

**El truco de reparametrización**: en lugar de muestrear $\mathbf{z} \sim q_\phi(\mathbf{z}|\mathbf{x}) = \mathcal{N}(\boldsymbol{\mu}, \text{diag}(\boldsymbol{\sigma}^2))$, escribimos:
$$\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}, \quad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0},\mathbf{I})$$
Esto mueve la aleatoriedad fuera del grafo computacional, permitiendo la retropropagación.

In [ ]:
class VAE_EEG(nn.Module):
    """
    VAE para segmentos de EEG/ECG 1-D.
    Encoder: x → (μ, log σ²)
    Decoder: z → x̂
    """
    def __init__(self, input_dim=250, latent_dim=16, hidden_dim=128):
        super().__init__()
        self.latent_dim = latent_dim

        # ── Encoder ──────────────────────────────────────────────────────────
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.fc_mu      = nn.Linear(hidden_dim, latent_dim)
        self.fc_log_var = nn.Linear(hidden_dim, latent_dim)

        # ── Decoder ──────────────────────────────────────────────────────────
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def encode(self, x):
        h       = self.encoder(x)
        mu      = self.fc_mu(h)
        log_var = self.fc_log_var(h)
        return mu, log_var

    def reparametrizar(self, mu, log_var):
        """z = μ + σ·ε, ε ~ N(0,I) — muestreo diferenciable."""
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)   # ε muestreado fuera del grafo
        return mu + std * eps

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z           = self.reparametrizar(mu, log_var)
        x_hat       = self.decode(z)
        return x_hat, mu, log_var

    def elbo_loss(self, x, x_hat, mu, log_var, beta=1.0):
        """ELBO = reconstrucción − β·KL"""
        recon_loss = F.mse_loss(x_hat, x, reduction='sum') / x.size(0)
        kl_loss    = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp()).sum(1).mean()
        return recon_loss + beta * kl_loss, recon_loss, kl_loss


# ── Simular dataset de latidos de ECG ─────────────────────────────────────────
def crear_latido_ecg(length=250, tipo_latido='N', rng_=None):
    rng_ = rng_ or np.random.default_rng()
    t = np.arange(length) / 250
    p  = 0.15 * np.exp(-((t-0.12)**2)/(2*0.015**2))
    r  = 1.00 * np.exp(-((t-0.22)**2)/(2*0.006**2))
    s  = -0.12 * np.exp(-((t-0.24)**2)/(2*0.008**2))
    tw = 0.25 * np.exp(-((t-0.38)**2)/(2*0.025**2))
    beat = p + r + s + tw
    if tipo_latido == 'V':
        beat = 0.9*np.exp(-((t-0.22)**2)/(2*0.018**2)) - \
               0.5*np.exp(-((t-0.28)**2)/(2*0.015**2)) + \
               0.2*np.exp(-((t-0.40)**2)/(2*0.030**2))
    return beat + rng_.normal(0, 0.03, length)

n_latidos = 1200
n_V       = 200
X_latidos_N = np.array([crear_latido_ecg(tipo_latido='N', rng_=rng) for _ in range(n_latidos)])
X_latidos_V = np.array([crear_latido_ecg(tipo_latido='V', rng_=rng) for _ in range(n_V)])
X_latidos   = np.vstack([X_latidos_N, X_latidos_V]).astype(np.float32)
y_latidos   = np.hstack([np.zeros(n_latidos), np.ones(n_V)]).astype(np.int64)

# Normalizar
X_latidos = (X_latidos - X_latidos.mean(1, keepdims=True)) / (X_latidos.std(1, keepdims=True)+1e-8)

Xt = torch.tensor(X_latidos, dtype=torch.float32)
yt = torch.tensor(y_latidos,  dtype=torch.long)
loader_vae = DataLoader(TensorDataset(Xt, yt), batch_size=64, shuffle=True)

print(f'Dataset ECG: {X_latidos.shape[0]} latidos, {X_latidos.shape[1]} muestras')

In [ ]:
# Entrenar el VAE con recocido (annealing) de β
vae = VAE_EEG(input_dim=250, latent_dim=8, hidden_dim=128).to(device)
opt_vae = optim.Adam(vae.parameters(), lr=1e-3)

n_epochs_vae = 60
hist_recon, hist_kl = [], []

for ep in range(n_epochs_vae):
    vae.train()
    ep_recon, ep_kl = 0, 0
    beta = min(1.0, ep / 20)   # recocido de KL: aumenta de 0 → 1 en 20 épocas

    for Xb, _ in loader_vae:
        Xb = Xb.to(device)
        x_hat, mu, log_var = vae(Xb)
        loss, rl, kl = vae.elbo_loss(Xb, x_hat, mu, log_var, beta=beta)
        opt_vae.zero_grad()
        loss.backward()
        opt_vae.step()
        ep_recon += rl.item(); ep_kl += kl.item()

    hist_recon.append(ep_recon / len(loader_vae))
    hist_kl.append(ep_kl    / len(loader_vae))
    if (ep+1) % 15 == 0:
        print(f'Época {ep+1:3d}  recon={hist_recon[-1]:.3f}  KL={hist_kl[-1]:.3f}  β={beta:.2f}')

# Curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_recon, 'b-', lw=2, label='Pérdida de reconstrucción')
axes[0].set(xlabel='Época', ylabel='MSE', title='Pérdida de reconstrucción del VAE')
axes[1].plot(hist_kl, 'r-', lw=2, label='Divergencia KL')
axes[1].set(xlabel='Época', ylabel='KL', title='Divergencia KL del VAE (recocida)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualización del espacio latente ─────────────────────────────────────────
vae.eval()
with torch.no_grad():
    mu_all, _ = vae.encode(Xt.to(device))
    mu_all    = mu_all.cpu().numpy()

pca_lat = PCA(n_components=2)
z_2d    = pca_lat.fit_transform(mu_all)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Espacio latente coloreado por clase
for cls, etiqueta, color in [(0,'Normal','steelblue'), (1,'PVC','tomato')]:
    mask = y_latidos == cls
    axes[0].scatter(z_2d[mask,0], z_2d[mask,1], alpha=0.4, s=12,
                     color=color, label=etiqueta)
axes[0].set(xlabel='PC1', ylabel='PC2', title='Espacio latente del VAE (PCA)\nSeparación N vs PVC')
axes[0].legend(fontsize=9)

# Calidad de reconstrucción
n_show = 4
x_samples = Xt[:n_show].to(device)
with torch.no_grad():
    x_recon, _, _ = vae(x_samples)
x_samples = x_samples.cpu().numpy()
x_recon   = x_recon.cpu().numpy()

t_axis = np.linspace(0, 1, 250)
for i in range(n_show):
    offset = i * 3
    axes[1].plot(t_axis, x_samples[i] + offset, 'navy', lw=1.2, alpha=0.7,
                  label='Original' if i==0 else '')
    axes[1].plot(t_axis, x_recon[i]  + offset, 'tomato', lw=1.5, ls='--',
                  label='Reconstruido' if i==0 else '')
axes[1].set(xlabel='Tiempo (s)', title='Reconstrucciones del VAE', yticks=[])
axes[1].legend(fontsize=8)

# Interpolación en el espacio latente: N → PVC
z_N   = mu_all[y_latidos == 0].mean(0)
z_PVC = mu_all[y_latidos == 1].mean(0)
alphas = np.linspace(0, 1, 6)

with torch.no_grad():
    for i, alpha in enumerate(alphas):
        z_interp = torch.tensor(
            (1-alpha)*z_N + alpha*z_PVC, dtype=torch.float32).unsqueeze(0).to(device)
        x_gen = vae.decode(z_interp).cpu().numpy().squeeze()
        axes[2].plot(t_axis, x_gen + i*2.5, lw=1.5,
                      color=plt.cm.RdBu_r(alpha), label=f'α={alpha:.1f}')

axes[2].set(xlabel='Tiempo (s)', title='Interpolación en el espacio latente\nNormal → PVC', yticks=[])
axes[2].legend(fontsize=7, loc='upper right')

plt.suptitle('VAE en latidos de ECG — exploración del espacio latente', y=1.01)
plt.tight_layout()
plt.show()

## Parte 2 — GAN condicional para aumentación de datos de EEG

El objetivo minimax de la GAN:
$$\min_G \max_D \; \mathbb{E}_{x\sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z\sim p_z}[\log(1-D(G(z)))]$$

**GAN condicional** añade la etiqueta de clase $y$ tanto a G como a D:
$$D(x,y), \quad G(z,y) \quad \Rightarrow \quad \text{genera muestras de la clase } y$$

In [ ]:
class cGAN_Generador(nn.Module):
    """Generador condicional: (z, etiqueta_clase) → latido de ECG sintético."""
    def __init__(self, z_dim=64, n_classes=2, output_dim=250):
        super().__init__()
        self.label_embed = nn.Embedding(n_classes, 16)
        self.net = nn.Sequential(
            nn.Linear(z_dim + 16, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 256),        nn.LeakyReLU(0.2),
            nn.Linear(256, 512),        nn.LeakyReLU(0.2),
            nn.Linear(512, output_dim), nn.Tanh(),
        )

    def forward(self, z, labels):
        cond = self.label_embed(labels)    # (B, 16)
        inp  = torch.cat([z, cond], dim=1) # (B, z_dim+16)
        return self.net(inp)


class cGAN_Discriminador(nn.Module):
    """Discriminador condicional: (x, etiqueta_clase) → puntuación real/falso."""
    def __init__(self, input_dim=250, n_classes=2):
        super().__init__()
        self.label_embed = nn.Embedding(n_classes, 16)
        self.net = nn.Sequential(
            nn.Linear(input_dim + 16, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(512, 256),            nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, x, labels):
        cond = self.label_embed(labels)
        inp  = torch.cat([x, cond], dim=1)
        return self.net(inp)


Z_DIM = 64
gen   = cGAN_Generador(z_dim=Z_DIM, n_classes=2).to(device)
disc  = cGAN_Discriminador(n_classes=2).to(device)

opt_G = optim.Adam(gen.parameters(),  lr=2e-4, betas=(0.5, 0.999))
opt_D = optim.Adam(disc.parameters(), lr=2e-4, betas=(0.5, 0.999))
crit_gan = nn.BCEWithLogitsLoss()

n_epochs_gan = 80
g_losses, d_losses = [], []

for ep in range(n_epochs_gan):
    ep_g, ep_d = 0, 0
    for Xb, yb in loader_vae:
        B  = Xb.size(0)
        Xb = Xb.to(device)
        yb = yb.to(device)

        real_lbl = torch.ones(B, 1, device=device)  * 0.9   # suavizado de etiquetas
        fake_lbl = torch.zeros(B, 1, device=device) + 0.1

        # ── Entrenar Discriminador ───────────────────────────────────────────
        z     = torch.randn(B, Z_DIM, device=device)
        fake  = gen(z, yb).detach()
        d_real = crit_gan(disc(Xb,   yb), real_lbl)
        d_fake = crit_gan(disc(fake, yb), fake_lbl)
        d_loss = (d_real + d_fake) / 2
        opt_D.zero_grad(); d_loss.backward(); opt_D.step()

        # ── Entrenar Generador ───────────────────────────────────────────────
        z     = torch.randn(B, Z_DIM, device=device)
        fake  = gen(z, yb)
        g_loss = crit_gan(disc(fake, yb), real_lbl)   # engañar a D
        opt_G.zero_grad(); g_loss.backward(); opt_G.step()

        ep_g += g_loss.item(); ep_d += d_loss.item()

    g_losses.append(ep_g / len(loader_vae))
    d_losses.append(ep_d / len(loader_vae))
    if (ep+1) % 20 == 0:
        print(f'Época {ep+1:3d}  G={g_losses[-1]:.4f}  D={d_losses[-1]:.4f}')

# Evaluar muestras generadas
gen.eval()
with torch.no_grad():
    z_test  = torch.randn(8, Z_DIM, device=device)
    lbl_N   = torch.zeros(4, dtype=torch.long, device=device)
    lbl_V   = torch.ones(4,  dtype=torch.long, device=device)
    gen_N   = gen(z_test[:4], lbl_N).cpu().numpy()
    gen_V   = gen(z_test[4:], lbl_V).cpu().numpy()

fig, axes = plt.subplots(3, 1, figsize=(13, 8))
axes[0].plot(g_losses, lw=2, label='Pérdida G'); axes[0].plot(d_losses, lw=2, label='Pérdida D')
axes[0].set(xlabel='Época', ylabel='Pérdida', title='Entrenamiento cGAN — pérdidas de G y D')
axes[0].legend()

t_ax = np.linspace(0, 1, 250)
for i in range(4):
    axes[1].plot(t_ax, X_latidos_N[i] + i*3, 'steelblue', lw=1, alpha=0.7)
    axes[1].plot(t_ax, gen_N[i]       + i*3, 'steelblue', lw=1.5, ls='--')
axes[1].set(yticks=[], title='Latidos Normales generados (punteado) vs reales (sólido)')

for i in range(4):
    axes[2].plot(t_ax, X_latidos_V[i] + i*3, 'tomato', lw=1, alpha=0.7)
    axes[2].plot(t_ax, gen_V[i]       + i*3, 'tomato', lw=1.5, ls='--')
axes[2].set(xlabel='Tiempo (s)', yticks=[], title='Latidos PVC generados (punteado) vs reales (sólido)')

plt.tight_layout()
plt.show()

## ✏️ Ejercicios

1. **Desenredo (disentanglement) con β-VAE.** Entrena el VAE de ECG con β ∈ {0.1, 1.0,
   4.0, 10.0}. Para cada β, recorre cada dimensión latente independientemente (varía una
   dimensión de −3σ a +3σ, manteniendo las demás en 0) y genera los latidos
   correspondientes. ¿Qué β produce las dimensiones latentes más interpretables
   (ej., una dimensión controlando el ancho del QRS, otra controlando la amplitud de R)?

2. **Detección de anomalías con VAE.** Usa el error de reconstrucción de un VAE entrenado
   en latidos Normales como puntuación de anomalía. Compara el AUROC para detección de
   PVC con (a) la densidad GMM de la Sesión 10 y (b) el SVM de una clase de la Sesión 7.
   Discute las ventajas relativas de cada enfoque.

3. **Wasserstein GAN.** Reemplaza la pérdida estándar de la GAN con la distancia de
   Wasserstein (WGAN-GP). Demuestra que el entrenamiento es más estable y se reduce el
   colapso de modo. Compara la calidad de las muestras generadas usando la distancia
   de Fréchet entre las distribuciones de características de latidos reales y generados.

4. **Aumentación con GAN para BCI.** Entrena una cGAN en datos de EEG de imaginería
   motora (simula o usa BCI Competition IV 2a). Genera 5× más muestras para la clase
   minoritaria, luego entrena un clasificador CNN en el dataset aumentado. Compara el
   AUROC con entrenar solo en datos reales, con tamaños de conjunto de entrenamiento
   equivalentes.

5. *(Desafío)* **Generación condicional con guía de clasificador.** Entrena un
   clasificador en los datos reales de ECG, luego usa su gradiente para guiar al decoder
   del VAE hacia la generación de muestras de una clase específica (classifier guidance /
   classifier-free guidance). Compara con el enfoque cGAN: ¿cuál produce muestras más
   realistas según un discriminador entrenado por separado?

## 📚 Conjuntos de datos

| Conjunto de datos | Fuente | Notas |
|---|---|
| MIT-BIH Arrhythmia | https://physionet.org/content/mitdb/ | Latidos de ECG para VAE/GAN |
| BCI Comp IV 2a | https://www.bbci.de/competition/iv/ | EEG de imaginería motora para aumentación con GAN |
| CINC 2017 | https://physionet.org/content/challenge-2017/ | ECG de una derivación, detección de FA |